In [ ]:
!pip install enoslib ipywidgets --break-system-packages

In [ ]:
!ssh rennes.grid5000.fr hostname

In [ ]:
!ssh-add ~/.ssh/id_ed25519

### G5K connection

Setup the `.python-grid5000.yaml` file with the username and password used to login to grid5000.

The file should look like this:
```yaml
username: G5K_LOGIN
password: G5K_password
```

-> required in order for the enoslib calls to g5k to work.

In addition to this, make sure to add these lines to your ssh configuration:

```text
Host g5k
    User G5K_LOGIN
    HostName access.grid5000.fr
    ForwardAgent no

Host !access.grid5000.fr *.grid5000.fr
    User G5K_LOGIN
    ProxyJump G5K_LOGIN@access.grid5000.fr
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host access.grid5000.fr
    User G5K_LOGIN
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host *.g5k
    User G5K_LOGIN
    ProxyCommand ssh g5k -W "$(basename %h .g5k):%p"
    ForwardAgent no
```

Make sure to replace `G5K_LOGIN` with your g5k username (the one from the site)

In [3]:
from grid5000 import Grid5000
import enoslib as en
import logging
import os
from datetime import datetime, timedelta

conf_file = os.path.join(os.environ.get("HOME"), ".python-grid5000.yaml")  # type: ignore
gk = Grid5000.from_yaml(conf_file)

# map each cluster to its site
cluster_to_site = {}
for site in gk.sites.list():
    for cluster in site.clusters.list():
        cluster_to_site[cluster.uid] = site.uid

# JOB CONFIGURATION
JOB_NAME = "fcquic_local_eval"
# CLUSTER="vianden"
CLUSTER = "larochette"
JOB_WALLTIME = timedelta(hours=3, minutes=0)

# usage policy check: the job cannot cross the day to night boundary at 7pm if it was submitted before 5 pm. Any job started after 5 pm can cross the boundary
# I had the issue once so this check is there to avoid receiving a usage policy violation email...
datetime_now = datetime.now()
job_end_dt = datetime_now + JOB_WALLTIME
if (
    datetime_now.hour <= 17 and job_end_dt.hour >= 19 and datetime_now.weekday() <= 5
):  # only check during weekdays
    raise RuntimeError(
        "This job reservation will violate the usage policy and will cross the day night boundary"
    )

# Display some general information about the library
en.check()
# Enable rich logging
_ = en.init_logging()

conf = en.G5kConf.from_settings(
    job_name=JOB_NAME,
    walltime=str(JOB_WALLTIME),
    env_name="debian13-nfs",  # using debian13 here, was 12 before
    job_type=["deploy"],
).add_machine(
    roles=["server"],
    servers=["larochette-4.luxembourg.grid5000.fr"],
    # cluster=CLUSTER,
    # nodes=1,
)

# This will validate the configuration, but not reserve resources yet
provider = en.G5k(conf)

_____        ___  ____  _ _ _
 | ____|_ __  / _ \/ ___|| (_) |__
 |  _| | '_ \| | | \___ \| | | '_ \
 | |___| | | | |_| |___) | | | |_) |
 |_____|_| |_|\___/|____/|_|_|_.__/  10.6.0

 • Documentation: ]8;id=630535;https://discovery.gitlabpages.inria.fr/enoslib/\https://discovery.gitlabpages.inria.fr/enoslib/]8;;\                            
 • Source: ]8;id=373049;https://gitlab.inria.fr/discovery/enoslib\https://gitlab.inria.fr/discovery/enoslib]8;;\                                         
 • Chat: ]8;id=759419;https://framateam.org/enoslib\https://framateam.org/enoslib]8;;\

                         Dependency check                         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider      ┃    Status     ┃ Hint                           ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Chameleon     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonKVM  │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonEdge │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Fabric        │ NOT INSTALLED │ pip install enoslib[fabric]    │
│ Distem        │ NOT INSTALLED │ pip install enoslib[distem]    │
│ IOT-lab       │ NOT INSTALLED │ pip install enoslib[iotlab]    │
│ Grid'5000     │   INSTALLED   │                                │
│ Openstack     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Vagrant       │ NOT INSTALLED │ pip install enoslib[vagrant]   │
│ VMonG5k       │   INSTALLED   │                                │
└───────────────┴───────────────┴────────────────────────────────┘

                                Connectivity check                                 
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider  ┃ Key                 ┃ Connectivity ┃ Hint                           ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Grid'5000 │ ssh:access          │      ✅      │ Connection to access.grid5000… │
│ Grid'5000 │ ssh:access:frontend │      ✅      │ Connection Host(rennes.grid50… │
│ Grid'5000 │ api:access          │      ✅      │                                │
│ VMonG5k   │ access              │      ❔      │ Check G5k status               │
└───────────┴─────────────────────┴──────────────┴────────────────────────────────┘

In [4]:
print("Reserving resources...")

# Get actual resources
roles, networks = provider.init()
display(roles)
display(networks)

# Fill in network information from nodes
roles = en.sync_info(roles, networks)

with en.actions(roles=roles, gather_facts=True) as a:
    a.apt(
        task_name="Install packages",
        name=[
            "tcpdump",
            "cmake",
            "clang",
            "python3.13-venv",
            "python-is-python3",
            "python3-pip",
            "btop",
            "htop",
        ],
        state="present",
    )

    # install frr
    a.file(
        task_name="Ensure apt keyring directory exists",
        path="/usr/share/keyrings",
        state="directory",
        mode="0755",
    )
    a.get_url(
        task_name="Download FRR GPG key",
        url="https://deb.frrouting.org/frr/keys.gpg",
        dest="/usr/share/keyrings/frrouting.gpg",
        mode="0644",
    )
    a.apt_repository(
        task_name="Add FRR apt repository",
        repo="deb [signed-by=/usr/share/keyrings/frrouting.gpg] https://deb.frrouting.org/frr {{ ansible_distribution_release }} frr-stable",
        filename="frr",
        state="present",
    )
    a.apt(
        task_name="Install FRR packages",
        name=["frr", "frr-pythontools"],
        state="present",
        update_cache=True,
    )

    # rust
    a.get_url(
        task_name="Download rustup installer",
        url="https://sh.rustup.rs",
        dest="/tmp/rustup-init.sh",
        mode="0755",
    )
    a.shell(
        task_name="Install Rust stable (rustup)",
        cmd="sh /tmp/rustup-init.sh -y --default-toolchain stable",
        creates="/root/.cargo/bin/rustup",
    )
    a.shell(
        task_name="Install Rust nightly toolchain",
        cmd="/root/.cargo/bin/rustup toolchain install nightly",
    )

    results = a.results

Reserving resources...


INFO     [G5k] Submitting {'name': 'fcquic_local_eval', 'types':         ]8;id=415520;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=663244;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#304\304]8;;\
         ['deploy', 'origin=enoslib_g5k'], 'resources':                                      
         "{network_address in ('larochette-4.luxembourg.grid5000.fr')}/n                     
         odes=1,walltime=3:00:00", 'command': 'sleep 31536000', 'queue':                     
         'default'} on luxembourg                                                            

INFO     [G5k] Waiting for 5 seconds before next OAR job(s) check...     ]8;id=255149;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=870612;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#337\337]8;;\

INFO     [G5k] Job 280536 on luxembourg: scheduled for 2026-05-13        ]8;id=667637;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=929146;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#345\345]8;;\
         20:10:02                                                                            

INFO     [G5k] Waiting for 10 seconds before next OAR job(s) check...    ]8;id=533675;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=375099;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#337\337]8;;\

...

Finished 7 tasks (Gathering Facts,setup,utils : include_tasks,utils : Dump network 
information in a file,utils : Create the fake interfaces) on 
{'larochette-4.luxembourg.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Finished 9 tasks (Gather facts,Install packages,Ensure apt keyring directory exists,Download 
FRR GPG key,Add FRR apt repository,Install FRR packages,Download rustup installer,Install 
Rust stable (rustup),Install Rust nightly toolchain) on 
{'larochette-4.luxembourg.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [5]:
# print os and kernel versions
from enoslib.api import Results

res: Results = en.run_command("uname -a", roles=roles)
print([res.stdout for res in res])

Output()

Finished 1 tasks (uname -a) on {'larochette-4.luxembourg.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

['Linux larochette-4.luxembourg.grid5000.fr 6.12.86+deb13-amd64 #1 SMP PREEMPT_DYNAMIC Debian 6.12.86-1 (2026-05-08) x86_64 GNU/Linux']


### Uploading project with SCP 

In [6]:
import subprocess

local_bin_dir = f"/home/corentin/fcquic_applications_master_thesis"
remote_bin_dir = "/tmp/chat"

for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        print(f"pushing binary to {host} (role: {role})")

        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/fcquic_chat"], check=True
        )
        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/evaluations"], check=True
        )
        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/multicast-quic"], check=True
        )

        print("pushing chat app")
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=logs/",
                "--exclude=baseline_logs/",
                "--exclude=tcp_logs/",
                "--exclude=tquic_logs/",
                "--exclude=target/",
                "--exclude=.git/",
                f"{local_bin_dir}/fcquic_chat/",
                f"{host}:{remote_bin_dir}/fcquic_chat/",
            ],
            check=True,
        )
        print("pushing multicast quic dir")
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=target/",
                "--exclude=.git/",
                f"{local_bin_dir}/multicast-quic/",
                f"{host}:{remote_bin_dir}/multicast-quic/",
            ],
            check=True,
        )
        print("pushing evaluation dir")
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=grid5000/",
                "--exclude=venv/",
                "--exclude=results/",
                "--exclude=graphs/",
                f"{local_bin_dir}/evaluations/",
                f"{host}:{remote_bin_dir}/evaluations/",
            ],
            check=True,
        )

        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/evaluations/graphs/"], check=True
        )
print("pushed project to node")

pushing binary to larochette-4.luxembourg.grid5000.fr (role: server)


pushing chat app


pushing multicast quic dir


pushing evaluation dir


pushed project to node


#### Rsync script.npf

In [59]:
import subprocess

for role, nodes in roles.items():
    for node in nodes:
        host = node.address

        # subprocess.run(
        #     [
        #         "rsync",
        #         "-az",
        #         "--exclude=logs/",
        #         "--exclude=baseline_logs/",
        #         "--exclude=tcp_logs/",
        #         "--exclude=tquic_logs/",
        #         "--exclude=target/",
        #         "--exclude=.git/",
        #         f"{local_bin_dir}/fcquic_chat/",
        #         f"{host}:{remote_bin_dir}/fcquic_chat/",
        #     ],
        #     check=True,
        # )

        # subprocess.run(
        #     [
        #         "rsync",
        #         "-az",
        #         f"{local_bin_dir}/evaluations/topologies/configs/latency/solo_10gbps.yaml",
        #         f"{host}:{remote_bin_dir}/evaluations/topologies/configs/latency/solo_10gbps.yaml",
        #     ],
        #     check=True,
        # )
        subprocess.run(
            [
                "rsync",
                "-az",
                f"{local_bin_dir}/evaluations/topologies/configs/latency/medium_5%_loss.yaml",
                f"{host}:{remote_bin_dir}/evaluations/topologies/configs/latency/medium_5%_loss.yaml",
            ],
            check=True,
        )

        subprocess.run(
            [
                "rsync",
                "-az",
                f"{local_bin_dir}/evaluations/tests/script.npf",
                f"{host}:{remote_bin_dir}/evaluations/tests/script.npf",
            ],
            check=True,
        )

        subprocess.run(
            [
                "rsync",
                "-az",
                f"{local_bin_dir}/evaluations/run_test.sh",
                f"{host}:{remote_bin_dir}/evaluations/run_test.sh",
            ],
            check=True,
        )

### Installing deps 

In [7]:
en.run_command(
    f"pip install graphviz networkx pandas brokenaxes --break-system-packages",
    roles=roles,
)

Output()

Finished 1 tasks (pip install graphviz networkx pandas brokenaxes --break-system-packages) on
{'larochette-4.luxembourg.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

[CommandResult(host='larochette-4.luxembourg.grid5000.fr', task='pip install graphviz networkx pandas brokenaxes --break-system-packages', status='OK', payload={'changed': True, 'stdout': 'Collecting graphviz\n  Downloading graphviz-0.21-py3-none-any.whl.metadata (12 kB)\nCollecting networkx\n  Downloading networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)\nCollecting pandas\n  Using cached pandas-3.0.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)\nCollecting brokenaxes\n  Downloading brokenaxes-0.6.2-py3-none-any.whl.metadata (6.6 kB)\nRequirement already satisfied: numpy>=1.26.0 in /usr/lib/python3/dist-packages (from pandas) (2.2.4)\nRequirement already satisfied: python-dateutil>=2.8.2 in /usr/lib/python3/dist-packages (from pandas) (2.9.0)\nRequirement already satisfied: matplotlib>3.6 in /usr/lib/python3/dist-packages (from brokenaxes) (3.10.1+dfsg1)\nRequirement already satisfied: contourpy>=1.0.1 in /usr/lib/python3/dist-packages (from matplotlib>3.6->brokenaxes) (1.3.1)\nRequirement already satisfied: cycler>=0.10 in /usr/lib/python3/dist-packages (from matplotlib>3.6->brokenaxes) (0.12.1)\nRequirement already satisfied: fonttools>=4.22.0 in /usr/lib/python3/dist-packages (from matplotlib>3.6->brokenaxes) (4.57.0)\nRequirement already satisfied: kiwisolver>=1.3.1 in /usr/lib/python3/dist-packages (from matplotlib>3.6->brokenaxes) (1.4.7)\nRequirement already satisfied: packaging>=20.0 in /usr/lib/python3/dist-packages (from matplotlib>3.6->brokenaxes) (25.0)\nRequirement already satisfied: pillow>=8 in /usr/lib/python3/dist-packages (from matplotlib>3.6->brokenaxes) (11.1.0)\nRequirement already satisfied: pyparsing>=2.3.1 in /usr/lib/python3/dist-packages (from matplotlib>3.6->brokenaxes) (3.1.2)\nDownloading graphviz-0.21-py3-none-any.whl (47 kB)\nDownloading networkx-3.6.1-py3-none-any.whl (2.1 MB)\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.0 MB/s eta 0:00:00\nUsing cached pandas-3.0.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)\nDownloading brokenaxes-0.6.2-py3-none-any.whl (7.3 kB)\nInstalling collected packages: pandas, networkx, graphviz, brokenaxes\n\nSuccessfully installed brokenaxes-0.6.2 graphviz-0.21 networkx-3.6.1 pandas-3.0.3', 'stderr': "WARNING: Running pip as the 'root' user can result in broken permissions and conflicting behaviour with the system package manager, possibly rendering your system unusable. It is recommended to use a virtual environment instead: https://pip.pypa.io/warnings/venv. Use the --root-user-action option if you know what you are doing and want to suppress this warning.", 'rc': 0, 'cmd': 'pip install graphviz networkx pandas brokenaxes --break-system-packages', 'start': '2026-05-13 20:44:54.748249', 'end': '2026-05-13 20:45:00.322564', 'delta': '0:00:05.574315', 'msg': '', 'invocation': {'module_args': {'_raw_params': 'pip install graphviz networkx pandas brokenaxes --break-system-packages', '_uses_shell': True, 'expand_argument_vars': True, 'stdin_add_newline': True, 'strip_empty_ends': True, 'argv': None, 'chdir': None, 'executable': None, 'creates': None, 'removes': None, 'stdin': None}}, 'stdout_lines': ['Collecting graphviz', '  Downloading graphviz-0.21-py3-none-any.whl.metadata (12 kB)', 'Collecting networkx', '  Downloading networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)', 'Collecting pandas', '  Using cached pandas-3.0.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)', 'Collecting brokenaxes', '  Downloading brokenaxes-0.6.2-py3-none-any.whl.metadata (6.6 kB)', 'Requirement already satisfied: numpy>=1.26.0 in /usr/lib/python3/dist-packages (from pandas) (2.2.4)', 'Requirement already satisfied: python-dateutil>=2.8.2 in /usr/lib/python3/dist-packages (from pandas) (2.9.0)', 'Requirement already satisfied: matplotlib>3.6 in /usr/lib/python3/dist-packages (from brokenaxes) (3.10.1+dfsg1)', 'Requirement already satisfied: contourpy>=1.0.1 in /usr/lib/python3/dist-packag

*Important:* SSH into the machine `root@vianden-1.luxembourg.grid5000.fr`, then run the following commands:
- `cd /tmp/chat/evaluations`
- `python -m venv venv`
- `source ./venv/bin/activate`
- `pip install npf`

In [ ]:
en.run_command(f"cd {remote_bin_dir}/fcquic_chat && cargo build --release", roles=roles)

### Test setup

In [67]:
# TEST_DIR_NAME = "latency"

# TOPO_CONF_NAME = "solo_10gbps"

# TOPO_CONF_NAME = "tiny_100mbps"
# TOPO_CONF_NAME = "tiny_1000mbps"

# TOPO_CONF_NAME = "small_0%_loss_1000mbps"
# TOPO_CONF_NAME = "small_0_1%_loss_1000mbps"
# TOPO_CONF_NAME = "small_0_5%_loss_1000mbps"
# TOPO_CONF_NAME = "small_5%_loss_1000mbps"
# TOPO_CONF_NAME = "small_10%_loss_1000mbps"

# TOPO_CONF_NAME = "medium_0%_loss"
# TOPO_CONF_NAME = "medium_0_5%_loss"
# TOPO_CONF_NAME = "medium_1%_loss"
# TOPO_CONF_NAME = "medium_5%_loss"

# TEST_DIR_NAME = "receivers"
# TOPO_CONF_NAME = "receivers"

TEST_DIR_NAME = "data"
TOPO_CONF_NAME = "data"

USE_POISSON = "true"
print("command to run")
print(f"bash run_test.sh {TEST_DIR_NAME} {TOPO_CONF_NAME} {USE_POISSON}")

command to run
bash run_test.sh data data true


### Running test script (NOTE: don't use this, it crashes the notebook)

In [ ]:
for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        subprocess.run(
            [
                "ssh",
                "root@" + host,
                f"cd {remote_bin_dir}/evaluations/ && bash run_test.sh {TEST_DIR_NAME} {TOPO_CONF_NAME} {USE_POISSON}",
            ],
            check=True,
        )

### Collecting data

In [61]:
import subprocess
import os

poisson_str = "poisson" if USE_POISSON else "uniform"
result_filename = f"npf_out_{TOPO_CONF_NAME}_{poisson_str}"

remote_out_dir = f"{remote_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/"
local_out_dir = f"{local_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/"

os.makedirs(local_out_dir, exist_ok=True)

for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        print(f"downloading results from {host}")
        subprocess.run(
            [
                "rsync",
                "-az",
                f"root@{host}:{remote_out_dir}{result_filename}.csv",
                f"root@{host}:{remote_out_dir}{result_filename}-TLOAD.csv",
                local_out_dir,
            ],
            check=True,
        )

print(f"results saved to {local_out_dir}{result_filename}")

downloading results from larochette-4.luxembourg.grid5000.fr


results saved to /home/corentin/fcquic_applications_master_thesis/evaluations/tests/latency/out/npf_out_medium_5%_loss_poisson


### Graph results

In [73]:
import subprocess

poisson_str = "poisson" if USE_POISSON else "uniform"
result_filename = f"npf_out_{TOPO_CONF_NAME}_{poisson_str}"
graphs_dir = f"{local_bin_dir}/evaluations/graphs"
csv_path = (
    f"{local_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/{result_filename}.csv"
)
output_dir = f"{graphs_dir}/{TEST_DIR_NAME}"

os.makedirs(output_dir, exist_ok=True)

args = ["python", f"{graphs_dir}/{TEST_DIR_NAME}.py", csv_path, output_dir]
# if TEST_DIR_NAME == "latency":
#     args.append("--inset")

args.append("--no-title")
subprocess.run(
    args,
    check=True,
)

print(f"graphs written to {output_dir}")

is poisson?: True
Outlier threshold: 1227787.975000002
Number of clients for this test: [61]
Additional data sizes tested: 1100-4400
Baseline QUIC samples: 0
Baseline TCP samples: 597266
Baseline TCP (NO TLS) samples: 620603
FC-QUIC samples: 267583
FC-QUIC with FEC samples: 0
Tokio-quiche samples: 615339
graphs written to /home/corentin/fcquic_applications_master_thesis/evaluations/graphs/data


### Compress results into a tarball

In [63]:
# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"{local_out_dir}results.tar.gz",
        f"{local_out_dir}{result_filename}.csv",
        f"{local_out_dir}{result_filename}-TLOAD.csv",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"{local_out_dir}results.tar.gz",
        output_dir,
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"{local_out_dir}{result_filename}.csv",
        f"{local_out_dir}{result_filename}-TLOAD.csv",
    ],
    check=True,
)

tar: Removing leading `/' from member names
tar: Removing leading `/' from hard link targets


CompletedProcess(args=['rm', '/home/corentin/fcquic_applications_master_thesis/evaluations/tests/latency/out/npf_out_medium_5%_loss_poisson.csv', '/home/corentin/fcquic_applications_master_thesis/evaluations/tests/latency/out/npf_out_medium_5%_loss_poisson-TLOAD.csv'], returncode=0)

#### Uncompress tarball into results (opposite of cell above)

In [69]:
# extract the tarball back into local_out_dir (inverse of the cell above)
local_out_dir = f"{local_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/"
output_dir = f"{graphs_dir}/{TEST_DIR_NAME}"
subprocess.run(
    [
        "cp",
        f"{output_dir}/results.tar.gz",
        local_out_dir,
    ],
    check=True,
)

subprocess.run(
    [
        "tar",
        "xzf",
        f"{local_out_dir}results.tar.gz",
        "-C",
        "/",
    ],
    check=True,
)

subprocess.run(
    [
        "rm",
        f"{local_out_dir}results.tar.gz",
    ],
    check=True,
)

CompletedProcess(args=['rm', '/home/corentin/fcquic_applications_master_thesis/evaluations/tests/data/out/results.tar.gz'], returncode=0)

### Stopping the current booking

In [64]:
provider.destroy()

INFO     [G5k] Reloading 280536 from luxembourg                          ]8;id=445449;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=481253;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#167\167]8;;\

INFO     [G5k] Killing the job (luxembourg, 280536)                      ]8;id=997259;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=480693;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#276\276]8;;\

INFO     [G5k] Job killed (luxembourg, 280536)                           ]8;id=220450;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=334483;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#257\257]8;;\